In [1]:
# !pip install git+https://github.com/huggingface/transformers.git qwen_vl_utils torchvision accelerate>=0.26.0
# pip install git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen_vl_utils torchvision
!pip install torch transformers 'accelerate>=0.26.0' bitsandbytes qwen_vl_utils torchvision

In [2]:
!nvidia-smi

Tue Aug 19 17:03:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:A6:00.0 Off |                    0 |
| N/A   29C    P0             78W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from simulations_core import *

In [ ]:
model_label = 'qwen2_7B'

model, processor = import_qwen2_7B()

generation_kwargs = {
    "max_new_tokens": 256,
    "do_sample": False,
    "top_k": 50,
    "top_p": 0.95,
    "temperature": 0.7,
    "output_scores": True,               # <--- Aggiunto
    "return_dict_in_generate": True,      # <--- Aggiunto
    "pad_token_id": processor.tokenizer.eos_token_id  # 👈 aggiungi questa riga
}

Using device: cuda


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

## CREATE GENERAL POOL OF IMAGES

In [ ]:
import os
import random
from tqdm import tqdm

# === Parametri ===
n_images = 100
square_size = 300
spacing = 100
dpi = 150
labels = ['A', 'B']

reference_dots = 10
difference_dots = 100  # es.: con 10 -> range [0, 110]

prob_threshold = 0.8

# Directory di salvataggio
output_dir = os.path.join("..", "images", f"images_dots_{model_label}")
os.makedirs(output_dir, exist_ok=True)

# Generazione immagini
fig_index = 0
attempts = 0
pbar = tqdm(total=n_images, desc="Generazione immagini valide")

# Prepara i bound per l'altro box
low = max(0, reference_dots - difference_dots)
high = reference_dots + difference_dots

if low == high == reference_dots:
    raise ValueError("difference_dots è 0: l'altro box avrebbe sempre lo stesso numero di punti del reference.")

while fig_index < n_images:
    position = random.randint(0, 1)
    correct_answer = labels[position]

    # Reference fisso
    n_reference = reference_dots

    # Campiona l'altro numero di punti nell'intervallo, evitando l'uguaglianza
    if low < high:
        n_other = random.randint(low, high)
        if n_other == n_reference:
            # spingi verso uno dei lati per garantire disuguaglianza
            if n_reference - low > high - n_reference and n_reference > 0:
                n_other = random.randint(low, max(low, n_reference - 1))
            else:
                n_other = random.randint(min(high, n_reference + 1), high)
    else:
        # Caso limite: intervallo di un solo valore diverso da reference (raro ma gestito)
        n_other = low
        if n_other == n_reference:
            # non dovrebbe capitare perché gestito sopra, ma per sicurezza:
            n_other = max(0, n_reference - 1)

    temp_path = os.path.join(output_dir, 'temp.png')

    # Crea immagine con pallini
    create_image_dots(
        n_reference, n_other, position, output_dir, 'temp.png',
        square_size=square_size, spacing=spacing, dpi=dpi
    )

    prompt = (
        f"In the image, there are three boxes labeled {labels[0]}, REFERENCE BOX, and {labels[1]}.\n"
        f"Which of the boxes, {labels[0]} or {labels[1]}, contains the same number of black dots as the REFERENCE BOX?\n"
        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
    )

    # Chiamata al modello (assumo single_query_qwen, cambia se usi un'altra family)
    prob_labels, output_scores = single_query_qwen(
        prompt, temp_path, labels, model, processor, generation_kwargs, print_flag=False
    )
    prob_correct = prob_labels.get(correct_answer, 0.0)

    image_name = f"dots_{fig_index}_{correct_answer}_p={prob_correct:.2f}_ref={n_reference}_other={n_other}.png"
    image_path = os.path.join(output_dir, image_name)

    # Verifica soglia
    if prob_correct >= prob_threshold:
        fig_index += 1
        pbar.update(1)
        os.rename(temp_path, image_path)
    else:
        try:
            os.remove(temp_path)
        except FileNotFoundError:
            pass

    attempts += 1

pbar.close()
print(f"\n✅ Completato: {n_images} immagini generate correttamente su {attempts} tentativi.")

## CREATE IMAGES FOR TASK DIFFICULTY AND PERFORMANCE

In [ ]:
# new ambiguity

# new ambiguity — DOTS

import os
import random
from tqdm import tqdm

# === Parametri immagine / task ===
square_size = 300
spacing = 100
dpi = 150
labels = ['A', 'B']

reference_dots = 10        # numero di punti nel box di riferimento
prob_threshold = 0.95      # soglia accettazione
n_images = 50              # immagini per ciascuna difference_dots

# elenco delle differenze da esplorare (modifica a piacere)
differences = list(range(20, 201, 20))  # 0, 20, 40, ..., 200

# cartella di output (unica)
output_dir = os.path.join("..", "images", f"images_dots_perplexity_diff_{model_label}")
os.makedirs(output_dir, exist_ok=True)

for difference_dots in differences:
    print(f"\n=== difference_dots = {difference_dots} ===")

    # range valido per l'altro box: [max(0, ref - diff), ref + diff]
    low = max(0, reference_dots - difference_dots)
    high = reference_dots + difference_dots

    # Pre-check: esiste almeno un intero nel range diverso da reference_dots?
    # (se range singoletto uguale a ref -> non generabile)
    if low == high == reference_dots:
        print(f"⚠️ Nessun range valido per difference_dots={difference_dots}. Salto.")
        continue

    fig_index = 0
    attempts = 0
    pbar = tqdm(total=n_images, desc=f"Generazione immagini (diff={difference_dots})")

    while fig_index < n_images:
        position = random.randint(0, 1)
        correct_answer = labels[position]

        # reference fisso
        n_reference = reference_dots

        # campiona n_other nel range, forzando != reference
        n_other = random.randint(low, high)
        if n_other == n_reference:
            # spingi verso un lato disponibile
            if n_reference > low:
                n_other = random.randint(low, n_reference - 1)
            elif n_reference < high:
                n_other = random.randint(n_reference + 1, high)
            else:
                attempts += 1
                continue

        temp_path = os.path.join(output_dir, "temp.png")

        # genera immagine
        create_image_dots(
            n_reference, n_other, position, output_dir, "temp.png",
            square_size=square_size, spacing=spacing, dpi=dpi
        )

        # prompt
        prompt = (
            f"In the image, there are three boxes labeled {labels[0]}, REFERENCE BOX, and {labels[1]}.\n"
            f"Which of the boxes, {labels[0]} or {labels[1]}, contains the same number of black dots as the REFERENCE BOX?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )

        # query al modello
        prob_labels, output_scores = single_query_qwen(
            prompt, temp_path, labels, model, processor, generation_kwargs, print_flag=False
        )
        prob_correct = prob_labels.get(correct_answer, 0.0)
        logit = output_scores.get(correct_answer, 0.0)

        # nome file (metadata: p, logit, diff, ref/other)
        image_name = (
            f"dots_{fig_index}_{correct_answer}"
            f"_p={prob_correct:.2f}_logit={logit:.2f}"
            f"_diff={difference_dots}_ref={n_reference}_other={n_other}.png"
        )
        image_path = os.path.join(output_dir, image_name)

        # accettazione / scarto
        if prob_correct >= prob_threshold:
            fig_index += 1
            pbar.update(1)
            os.rename(temp_path, image_path)
        else:
            try:
                os.remove(temp_path)
            except FileNotFoundError:
                pass

        attempts += 1

    pbar.close()
    print(f"✅ Completato diff={difference_dots}: {n_images} immagini su {attempts} tentativi.")
